# NB4: Parser (NL → ActionIntent)

**Purpose:** Test that the parser LLM can reliably convert natural language player commands into structured ActionIntents.

**Tests:**
1. 10 diverse natural language commands parsed successfully
2. Instruments in output exist in actor's inventory
3. Action category correctly inferred
4. Rejection test: parser handles commands using instruments the actor doesn't have
5. Ambiguity test: vague commands produce ambiguity_flags

**Pass criteria:** >90% parse success, zero hallucinated instrument IDs.

In [ ]:
import sys
sys.path.insert(0, '..')

import uuid
from wargame.scenario import load_scenario
from wargame.models import ActionIntent
from wargame.parser import build_parser_messages, validate_action_intent
from llm_client import call_llm_structured

SCENARIO_PATH = '../scenarios/us_iran_2026.yaml'
PARSER_MODEL = 'gemini/gemini-2.5-flash'
TRACE_ID = f'nb4_parser_{uuid.uuid4().hex[:8]}'

spec = load_scenario(SCENARIO_PATH)
us = next(a for a in spec.actors if a.id == 'actor_us')
iran = next(a for a in spec.actors if a.id == 'actor_iran')

print(f'US instruments: {[i.id for i in us.instruments]}')
print(f'Iran instruments: {[i.id for i in iran.instruments]}')

## Define 10 Test Directives

A mix of specificity levels, both actors, and edge cases.

In [ ]:
test_directives = [
    # US actions — clear
    ('actor_us', 'Sanction their central bank. Hit them with everything we have on the financial side.'),
    ('actor_us', 'Send the Lincoln carrier group to the Persian Gulf. Make sure they know we\'re there.'),
    ('actor_us', 'Open diplomatic talks through the Swiss embassy. Feel them out on enrichment limits.'),
    ('actor_us', 'Get Cyber Command to go after their centrifuge controllers. Stuxnet 2.0. Zero attribution.'),
    ('actor_us', 'Launch a social media campaign exposing regime corruption among senior IRGC commanders.'),
    
    # Iran actions — clear
    ('actor_iran', 'Tell Hezbollah to increase rocket attacks on northern Israel. Show them we have reach.'),
    ('actor_iran', 'Speed up enrichment at Fordow. Get to 60% as fast as possible.'),
    ('actor_iran', 'Reach out to the Europeans through back channels. See if they\'ll break from Washington.'),
    
    # Vague commands — should produce ambiguity flags
    ('actor_us', 'Apply maximum pressure. Use everything available.'),
    
    # Tricky — mentions capability the actor doesn't have
    ('actor_iran', 'Launch a cyberattack on US financial infrastructure.'),
]

print(f'Defined {len(test_directives)} test directives')

In [ ]:
results = []

for i, (actor_id, directive) in enumerate(test_directives):
    actor = us if actor_id == 'actor_us' else iran
    instruments = [{'id': inst.id, 'midfield': inst.midfield, 'target_vars': inst.target_vars} for inst in actor.instruments]
    budget = spec.resource_budget[actor_id].domains
    
    messages = build_parser_messages(directive, actor, instruments, budget)
    
    print(f'\n--- Directive {i+1}/{len(test_directives)} [{actor_id}] ---')
    print(f'  "{directive[:80]}..."' if len(directive) > 80 else f'  "{directive}"')
    
    try:
        intent, call_result = call_llm_structured(
            model=PARSER_MODEL,
            messages=messages,
            response_model=ActionIntent,
            task='wargame_parser',
            trace_id=TRACE_ID,
            max_budget=0.5,
        )
        
        issues = validate_action_intent(intent, actor)
        
        results.append({
            'directive_idx': i,
            'actor_id': actor_id,
            'directive': directive,
            'intent': intent,
            'issues': issues,
            'success': len(issues) == 0,
        })
        
        print(f'  Category: {intent.action_category}')
        print(f'  Instruments: {intent.instruments_used}')
        print(f'  Effect: {intent.intended_effect[:80]}...')
        print(f'  Cost: {intent.resource_cost}')
        if intent.ambiguity_flags:
            print(f'  Ambiguity flags: {intent.ambiguity_flags}')
        if issues:
            print(f'  ⚠ Issues: {issues}')
        else:
            print(f'  ✓ Valid')
            
    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        results.append({
            'directive_idx': i,
            'actor_id': actor_id,
            'directive': directive,
            'intent': None,
            'issues': [str(e)],
            'success': False,
        })

successes = sum(1 for r in results if r['success'])
print(f'\n=== Results: {successes}/{len(results)} valid parses ===')
print(f'Parse rate: {successes/len(results)*100:.0f}%')

## Detailed Analysis

In [ ]:
# Check instrument validity across all results
all_instruments_valid = True
for r in results:
    if r['intent'] is None:
        continue
    actor = us if r['actor_id'] == 'actor_us' else iran
    valid_ids = {inst.id for inst in actor.instruments}
    for inst_id in r['intent'].instruments_used:
        if inst_id not in valid_ids:
            print(f'⚠ Hallucinated instrument: {inst_id} (directive {r["directive_idx"]}: {r["directive"][:50]}...)')
            all_instruments_valid = False

if all_instruments_valid:
    print('✓ Zero hallucinated instrument IDs')

# Check ambiguity detection
vague_directive_idx = 8  # "Apply maximum pressure. Use everything available."
vague_result = results[vague_directive_idx]
if vague_result['intent'] and vague_result['intent'].ambiguity_flags:
    print(f'✓ Vague directive detected ambiguity: {vague_result["intent"].ambiguity_flags}')
else:
    print(f'⚠ Vague directive did not produce ambiguity flags')

# Check the tricky directive (Iran cyber attack — no cyber instrument)
tricky_idx = 9
tricky = results[tricky_idx]
if tricky['issues']:
    print(f'✓ Tricky directive correctly flagged: {tricky["issues"]}')
elif tricky['intent']:
    print(f'⚠ Tricky directive: parser mapped to {tricky["intent"].instruments_used} — check if valid')
    # Iran doesn't have inst_iran_cyber, so if parser picked something else, that might be creative
    actor = iran
    valid_ids = {inst.id for inst in actor.instruments}
    for inst_id in tricky['intent'].instruments_used:
        if inst_id in valid_ids:
            print(f'  Parser creatively mapped to available instrument: {inst_id}')

In [ ]:
# Cost analysis
try:
    from llm_client import get_cost
    cost = get_cost(trace_id=TRACE_ID)
    print(f'Total cost for {len(test_directives)} parse calls: ${cost:.4f}')
    print(f'Cost per parse: ${cost/len(test_directives):.4f}')
except Exception as e:
    print(f'Cost query: {e}')

## Summary

**Pass criteria:**
- [ ] >90% parse to valid ActionIntent
- [ ] Zero hallucinated instrument IDs
- [ ] Action categories correctly inferred
- [ ] Vague command produces ambiguity_flags
- [ ] Parser handles missing instruments gracefully